# Flash Attention V2 课后练习：实现 Causal Mask

本练习要求你修改 Flash Attention V2 的实现，支持**因果 Attention**（Causal Attention），这是 GPT 等 autoregressive 模型训练时的核心需求。

## 任务要求

1. 正确构建下三角 mask
2. 在计算注意力分数时应用 mask
3. 验证与 PyTorch 的 `F.scaled_dot_product_attention(..., is_causal=True)` 结果一致

## 背景

在 Decoder 场景下，训练时需要**因果性**：当前 token 只能看到过去的 token，不能"偷看"未来。

数学上，这相当于在注意力分数矩阵上应用一个下三角 mask：

$$
S_{masked}[i, j] = \begin{cases}
S[i, j] & \text{if } j \leq i \\
-\infty & \text{if } j > i
\end{cases}
$$

这样当 $j > i$ 时，$\exp(S_{masked}[i, j]) = \exp(-\infty) = 0$，未来位置就被"屏蔽"了。

## 提示

### 1. Mask 位置

在计算 `S = Q @ K^T` 之后，Softmax 之前应用 mask。

### 2. Mask 构建逻辑

可以用 `k_pos > q_pos` 判断是否需要 mask：

```python
q_pos: 当前 Q block 的 query 位置（绝对位置）
k_pos: 当前 K block 的 key 位置（绝对位置）

# mask[i, j] = (k_pos[j] > q_pos[i]) ? -inf : 0
```

### 3. 注意点

- `q_pos` 和 `k_pos` 需要使用**绝对位置**（不是 block 内的相对位置）
- Mask 的值应该是足够小的负数（如 `-inf`），这样 `exp(-inf) = 0`

## 练习

请在下面的 `TODO` 位置填入你的实现：

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def flash_attn_v2_causal_kernel(
    q_ptr, k_ptr, v_ptr,
    o_ptr,
    stride_qz, stride_qh, stride_qm, stride_qk,
    stride_kz, stride_kh, stride_kn, stride_kk,
    stride_vz, stride_vh, stride_vn, stride_vk,
    stride_oz, stride_oh, stride_om, stride_ok,
    batch, heads,
    seqlen_q, seqlen_k,
    head_dim,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    """Flash Attention V2 with Causal Mask"""
    # 计算 program_id
    n_q_blocks = (seqlen_q + BLOCK_M - 1) // BLOCK_M
    pid = tl.program_id(axis=0)

    q_block_id = pid % n_q_blocks
    pid_ = pid // n_q_blocks
    head_id = pid_ % heads
    batch_id = pid_ // heads

    # 加载 Q block
    q_ptr += batch_id * stride_qz + head_id * stride_qh
    m_mask = q_block_id * BLOCK_M + tl.arange(0, BLOCK_M) < seqlen_q
    q_ptr += (q_block_id * BLOCK_M + tl.arange(0, BLOCK_M))[:, None] * stride_qm
    q = tl.load(
        q_ptr + tl.arange(0, head_dim)[None, :] * stride_qk,
        mask=m_mask[:, None],
        other=0.0
    )

    # 初始化累加器
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32) + 1.0
    acc = tl.zeros([BLOCK_M, head_dim], dtype=tl.float32)
    sm_scale = 1.0 / (head_dim ** 0.5)

    # K/V 的起始地址
    k_ptr += batch_id * stride_kz + head_id * stride_kh
    v_ptr += batch_id * stride_vz + head_id * stride_vh

    # 遍历所有 K/V block
    lo = 0
    hi = (seqlen_k + BLOCK_N - 1) // BLOCK_N

    for k_block_id in range(lo, hi):
        n_mask = k_block_id * BLOCK_N + tl.arange(0, BLOCK_N) < seqlen_k

        # 加载 K block
        k_ptr_offset = (k_block_id * BLOCK_N + tl.arange(0, BLOCK_N))[None, :] * stride_kn
        k = tl.load(
            k_ptr + k_ptr_offset + tl.arange(0, head_dim)[:, None] * stride_kk,
            mask=n_mask[None, :],
            other=0.0
        )

        # 加载 V block
        v_ptr_offset = (k_block_id * BLOCK_N + tl.arange(0, BLOCK_N))[None, :] * stride_vn
        v = tl.load(
            v_ptr + v_ptr_offset + tl.arange(0, head_dim)[:, None] * stride_vk,
            mask=n_mask[None, :],
            other=0.0
        )

        # 计算 Q @ K^T
        qk = tl.dot(q, k) * sm_scale

        # ============ TODO: 在这里添加因果 mask ============
        # 1. 计算 q_pos（query 的绝对位置）
        # 2. 计算 k_pos（key 的绝对位置）
        # 3. 构建 mask: k_pos > q_pos 的位置设为 -inf
        # 4. 应用 mask: qk = qk + mask
        # ==================================================

        # 在线 Softmax 更新
        m_block = tl.max(qk, axis=1)
        m_i_new = tl.maximum(m_i, m_block)
        alpha = tl.exp(m_i - m_i_new)
        p = tl.exp(qk - m_i_new[:, None])
        l_i_new = alpha * l_i + tl.sum(p, axis=1)
        acc = alpha * acc + tl.dot(p.to(q.dtype), v)
        m_i, l_i = m_i_new, l_i_new

    # 写回结果
    acc = acc / l_i[:, None]
    o_ptr += batch_id * stride_oz + head_id * stride_oh
    o_ptr += (q_block_id * BLOCK_M + tl.arange(0, BLOCK_M))[:, None] * stride_om
    tl.store(
        o_ptr + tl.arange(0, head_dim)[None, :] * stride_ok,
        acc,
        mask=m_mask[:, None]
    )


def flash_attn_v2_causal(q, k, v):
    """Flash Attention V2 with Causal Mask"""
    batch, heads, seqlen_q, head_dim = q.shape
    seqlen_k = k.shape[2]

    n_q_blocks = (seqlen_q + 127) // 128
    grid = (batch * heads * n_q_blocks,)

    o = torch.empty_like(q)

    flash_attn_v2_causal_kernel[grid](
        q, k, v, o,
        *q.stride(), *k.stride(), *v.stride(), *o.stride(),
        batch, heads, seqlen_q, seqlen_k, head_dim,
        BLOCK_M=128, BLOCK_N=128,
    )

    return o

## 验证

完成实现后，运行下面的代码验证正确性：

In [ ]:
import torch.nn.functional as F

def test_causal_correctness():
    """验证因果 Attention 的正确性"""
    print("=" * 60)
    print("因果 Attention 正确性验证")
    print("=" * 60)

    # 测试不同配置
    test_cases = [
        (1, 2, 256, 256, 64),
        (1, 4, 512, 512, 64),
        (2, 4, 1024, 1024, 64),
    ]

    for batch, heads, seqlen, head_dim, d in test_cases:
        print(f"\n测试配置: batch={batch}, heads={heads}, seqlen={seqlen}, head_dim={d}")

        q = torch.randn(batch, heads, seqlen, d, device='cuda', dtype=torch.float16)
        k = torch.randn(batch, heads, seqlen, d, device='cuda', dtype=torch.float16)
        v = torch.randn(batch, heads, seqlen, d, device='cuda', dtype=torch.float16)

        # 你的实现
        o_mine = flash_attn_v2_causal(q, k, v)

        # PyTorch 参考实现
        o_torch = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # 验证
        max_error = torch.max(torch.abs(o_mine - o_torch)).item()
        mean_error = torch.mean(torch.abs(o_mine - o_torch)).item()

        print(f"  Max error: {max_error:.6f}")
        print(f"  Mean error: {mean_error:.6f}")

        if torch.allclose(o_mine, o_torch, atol=1e-2):
            print("  ✓ 通过")
        else:
            print("  ✗ 失败")

    print("\n" + "=" * 60)

if torch.cuda.is_available():
    test_causal_correctness()
else:
    print("CUDA 不可用，跳过测试")

## 参考答案

如果你完成了练习，可以查看下面的参考答案对比一下：

In [ ]:
# ============ 参考答案 ============

# 在 `qk = tl.dot(q, k) * sm_scale` 之后添加以下代码：

# 计算 query 和 key 的绝对位置
q_pos = q_block_id * BLOCK_M + tl.arange(0, BLOCK_M)
k_pos = k_block_id * BLOCK_N + tl.arange(0, BLOCK_N)

# 构建因果 mask: k_pos > q_pos 的位置设为 -inf
# mask[i, j] = -inf if k_pos[j] > q_pos[i], else 0
mask = tl.where(
    k_pos[None, :] > q_pos[:, None],
    float(-"inf"),
    0.0
)

# 应用 mask
qk = qk + mask
# ===================================